# SU(4) persistent pipeline
Run the single cell. Upload only the full symbolic source ZIP. The pipeline regenerates both SU(4) result bundles, saves them to Google Drive, creates one master ZIP, and downloads it immediately.

In [ ]:
#!/usr/bin/env python3
"""
Persistent one-run SU(4) pipeline.

This fixes the prior ephemeral-/content failure by:
1. uploading the full symbolic source bundle;
2. running the SU(4) exceptional enumerator;
3. running the SU(4) local algebra;
4. creating one master result ZIP;
5. copying all results to Google Drive;
6. immediately downloading the master ZIP.

The output remains available in Drive even if the Colab runtime resets.
"""
from __future__ import annotations

import contextlib
import hashlib
import json
import os
import runpy
import shutil
import sys
import time
import zipfile
from pathlib import Path

BASE = Path("/content")
WORK = BASE / "SU4_PERSISTENT_PIPELINE_V1"
WORK.mkdir(parents=True, exist_ok=True)
LOG_PATH = WORK / "SU4_PERSISTENT_PIPELINE_V1.log"

ENUM_SOURCE = '#!/usr/bin/env python3\n"""\nSU(4) exceptional-rank O(y^4) corpus enumerator and resolvent-impact audit.\n\nThis is the first exact SU(4) stage. It reuses the verified group-independent\nStage-0 geometry and the stable-rank Stage-1 sign basis, replacing only\n\n    local charge == 0\n\nby the exact SU(4) N-ality condition\n\n    local charge == 0 mod 4.\n\nUnlike SU(6), SU(4) determinant sectors can occur at the three resolvent cuts.\nThis script therefore does not attempt the contraction. It enumerates the full\nexceptional word/sign corpus and classifies, exactly:\n\n  * unchanged stable-rank regression;\n  * new and mixed SU(4) words;\n  * final local Haar families (4,0), (0,4), (5,1), (1,5);\n  * every cut at which a pure determinant channel Lambda^4 V occurs;\n  * canonical local signatures needed by the finite-rank contraction engine.\n\nInput:\n  Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE_2026-06-14_V2*.zip\n\nOutputs:\n  SU4_EXCEPTIONAL_ENUMERATOR_V1/SU4_EXCEPTIONAL_ENUMERATOR_V1.json\n  SU4_EXCEPTIONAL_ENUMERATOR_V1/SU4_EXCEPTIONAL_ENUMERATOR_V1.md\n  SU4_EXCEPTIONAL_ENUMERATOR_V1/y4_su4_ordered_words.json.gz\n  SU4_EXCEPTIONAL_ENUMERATOR_V1/y4_su4_exceptional_only_words.json.gz\n  SU4_EXCEPTIONAL_ENUMERATOR_V1/y4_su4_local_signature_catalog.json\n  SU4_EXCEPTIONAL_ENUMERATOR_V1_BUNDLE.zip\n"""\nfrom __future__ import annotations\n\nimport gzip\nimport hashlib\nimport importlib.util\nimport itertools\nimport json\nimport os\nimport re\nimport sys\nimport time\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nVERSION = "2026-06-14-su4-exceptional-enumerator-v1"\nN = 4\nBASE = Path("/content") if Path("/content").exists() else Path("/mnt/data")\nOUT = BASE / "SU4_EXCEPTIONAL_ENUMERATOR_V1"\nEXTRACT = OUT / "extracted"\nOUT.mkdir(parents=True, exist_ok=True)\nEXTRACT.mkdir(parents=True, exist_ok=True)\n\nPREFERRED_BUNDLE_GLOB = "Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE*.zip"\nREQUIRED_STAGE1 = "y4_sun_stable_rank_stage1.py"\nREQUIRED_WORDS = "y4_sun_stable_ordered_words.json.gz"\n\n\ndef gate(name: str, cond: bool, detail: str = "") -> None:\n    status = "PASS" if cond else "FAIL"\n    print(f"{status:4s} {name:88s} {detail}")\n    if not cond:\n        raise AssertionError(f"{name}: {detail}")\n\n\ndef sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for block in iter(lambda: f.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef write_json(path: Path, payload: Any) -> str:\n    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")\n    return sha256(path)\n\n\ndef write_json_gz(path: Path, payload: Any) -> str:\n    raw = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")\n    with path.open("wb") as raw_file:\n        with gzip.GzipFile(filename="", mode="wb", compresslevel=9, fileobj=raw_file, mtime=0) as f:\n            f.write(raw)\n    return sha256(path)\n\n\ndef read_json_gz(path: Path) -> Any:\n    with gzip.open(path, "rt", encoding="utf-8") as f:\n        return json.load(f)\n\n\ndef safe_extract(zpath: Path, dest: Path) -> list[Path]:\n    dest.mkdir(parents=True, exist_ok=True)\n    root = dest.resolve()\n    with zipfile.ZipFile(zpath) as zf:\n        for info in zf.infolist():\n            target = (dest / info.filename).resolve()\n            if target != root and not str(target).startswith(str(root) + os.sep):\n                raise ValueError(f"unsafe ZIP member: {info.filename}")\n        zf.extractall(dest)\n        return [dest / i.filename for i in zf.infolist() if not i.is_dir()]\n\n\ndef recursive_extract(archives: Iterable[Path], max_depth: int = 4) -> list[dict[str, Any]]:\n    queue = [(Path(p), 0) for p in archives]\n    seen: set[str] = set()\n    records: list[dict[str, Any]] = []\n    while queue:\n        zp, depth = queue.pop(0)\n        if depth > max_depth or not zp.is_file():\n            continue\n        try:\n            h = sha256(zp)\n        except Exception:\n            continue\n        if h in seen:\n            continue\n        seen.add(h)\n        label = re.sub(r"[^A-Za-z0-9_.-]+", "_", zp.stem)[:90]\n        dest = EXTRACT / f"d{depth}_{label}_{h[:10]}"\n        try:\n            files = safe_extract(zp, dest)\n            records.append({\n                "archive": str(zp), "sha256": h, "depth": depth,\n                "destination": str(dest), "file_count": len(files), "status": "ok",\n            })\n            for p in files:\n                if p.suffix.lower() == ".zip":\n                    queue.append((p, depth + 1))\n        except Exception as exc:\n            records.append({\n                "archive": str(zp), "sha256": h, "depth": depth,\n                "status": "error", "error": repr(exc),\n            })\n    return records\n\n\ndef find_unique(name: str, roots: Iterable[Path]) -> list[Path]:\n    found: dict[str, Path] = {}\n    for root in roots:\n        if not root.exists():\n            continue\n        try:\n            for p in root.rglob(name):\n                if p.is_file():\n                    found[sha256(p)] = p\n        except Exception:\n            pass\n    return sorted(found.values(), key=lambda p: str(p))\n\n\ndef find_glob(pattern: str, roots: Iterable[Path]) -> list[Path]:\n    found: dict[str, Path] = {}\n    for root in roots:\n        if not root.exists():\n            continue\n        try:\n            for p in root.rglob(pattern):\n                if p.is_file():\n                    found[sha256(p)] = p\n        except Exception:\n            pass\n    return sorted(found.values(), key=lambda p: str(p))\n\n\ndef upload_if_needed() -> list[Path]:\n    roots = [BASE]\n    candidates = find_glob(PREFERRED_BUNDLE_GLOB, roots)\n    candidates += find_unique(REQUIRED_STAGE1, roots)\n    candidates += find_unique(REQUIRED_WORDS, roots)\n    if candidates:\n        return candidates\n    if not Path("/content").exists():\n        return []\n    try:\n        from google.colab import files as colab_files  # type: ignore\n    except Exception:\n        return []\n    print("\\nUPLOAD REQUIRED — select the full symbolic source bundle:")\n    print("  Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE_2026-06-14_V2*.zip")\n    uploaded = colab_files.upload()\n    saved: list[Path] = []\n    for name, data in uploaded.items():\n        target = Path("/content") / Path(name).name\n        target.write_bytes(data)\n        saved.append(target)\n        print(f"saved {len(data):,} bytes -> {target}")\n    return saved\n\n\ndef load_module(name: str, path: Path):\n    spec = importlib.util.spec_from_file_location(name, str(path))\n    if spec is None or spec.loader is None:\n        raise ImportError(path)\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[name] = module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef bit_indices(mask: int) -> list[int]:\n    return [i for i in range(64) if (mask >> i) & 1]\n\n\ndef key_from_record(record: dict[str, Any]):\n    return (\n        tuple(tuple(int(x) for x in p) for p in record["ordered_insertions"]),\n        tuple(int(x) for x in record["output"]),\n    )\n\n\ndef assignments_from_record(record: dict[str, Any]) -> set[tuple[int, ...]]:\n    return {tuple(int(x) for x in s) for s in record["exact_balance_assignments"]}\n\n\ndef c_representative(signs: tuple[int, ...]) -> tuple[int, ...]:\n    conjugate = tuple(-x for x in signs)\n    return min(signs, conjugate)\n\n\ndef signature_representative(sig: tuple[int, ...]) -> tuple[int, ...]:\n    conjugate = tuple(-x for x in sig)\n    return min(sig, conjugate)\n\n\ndef main() -> None:\n    started = time.time()\n    print("=" * 116)\n    print("SU(4) O(y^4) EXCEPTIONAL-RANK CORPUS ENUMERATOR + RESOLVENT AUDIT")\n    print("=" * 116)\n    print("version :", VERSION)\n    print("output  :", OUT)\n    print("hardware: CPU exact combinatorics; GPU not used")\n\n    initial = upload_if_needed()\n    archives = [p for p in initial if p.suffix.lower() == ".zip"]\n    archives += find_glob(PREFERRED_BUNDLE_GLOB, [BASE])\n    extraction = recursive_extract(archives)\n    roots = [BASE, EXTRACT]\n\n    stage1_paths = find_unique(REQUIRED_STAGE1, roots)\n    words_paths = find_unique(REQUIRED_WORDS, roots)\n    gate("stable-rank Stage-1 source located", bool(stage1_paths), str(stage1_paths[:3]))\n    gate("4,171-word stable archive located", bool(words_paths), str(words_paths[:3]))\n\n    st_path = stage1_paths[0]\n    words_path = words_paths[0]\n    st = load_module("y4_sun_stable_rank_stage1_su4_enum", st_path)\n\n    required_names = [\n        "locate_complete_source", "decode_stage1", "load_module_from_source",\n        "ensure_stage0_supports", "rows_for", "SIGNS", "FULL_MASK",\n    ]\n    missing = [name for name in required_names if not hasattr(st, name)]\n    gate("stable Stage-1 runtime contract available", not missing, str(missing))\n    gate("stable sign basis contains 64 assignments", len(st.SIGNS) == 64, str(len(st.SIGNS)))\n\n    complete_source = st.locate_complete_source(None)\n    stage0_source, stage1_source = st.decode_stage1(complete_source)\n    stage0 = st.load_module_from_source("y4_stage0_su4_enum", stage0_source)\n    stage1_local = st.load_module_from_source("y4_stage1_su4_enum", stage1_source)\n    support_path = Path(st.ensure_stage0_supports(stage0, stage0_source, None))\n    support_payload = read_json_gz(support_path)\n    supports = [\n        tuple(tuple(int(x) for x in p) for p in record)\n        for record in support_payload["supports"]\n    ]\n    gate("Stage-0 connected support count", len(supports) == 182440, f"{len(supports):,}")\n\n    signs_basis = tuple(tuple(int(x) for x in s) for s in st.SIGNS)\n    full_mask = int(st.FULL_MASK)\n    gate("FULL_MASK covers all 64 assignments", full_mask == (1 << 64) - 1, hex(full_mask))\n\n    row_cache_stable: dict[tuple[int, ...], int] = {}\n    row_cache_su4: dict[tuple[int, ...], int] = {}\n    max_local_degree = 0\n\n    def row_mask(row: tuple[int, ...], *, su4: bool) -> int:\n        nonlocal max_local_degree\n        cache = row_cache_su4 if su4 else row_cache_stable\n        if row in cache:\n            return cache[row]\n        max_local_degree = max(max_local_degree, sum(abs(int(x)) for x in row))\n        mask = 0\n        for index, signs in enumerate(signs_basis):\n            charge = sum(int(row[j]) * int(signs[j]) for j in range(6))\n            ok = (charge % N == 0) if su4 else (charge == 0)\n            if ok:\n                mask |= 1 << index\n        cache[row] = mask\n        return mask\n\n    def mask_for(word, output, *, su4: bool) -> int:\n        mask = full_mask\n        for row0 in st.rows_for(stage0, word, output):\n            row = tuple(int(x) for x in row0)\n            mask &= row_mask(row, su4=su4)\n            if mask == 0:\n                break\n        return mask\n\n    stable_survivors: dict[Any, int] = {}\n    su4_survivors: dict[Any, int] = {}\n    candidate_pairs = 0\n\n    for index, multiset in enumerate(supports, start=1):\n        for output in stage0.candidate_outputs(multiset):\n            candidate_pairs += 1\n            stable_mask = mask_for(multiset, output, su4=False)\n            su4_mask = mask_for(multiset, output, su4=True)\n            if stable_mask:\n                stable_survivors[stage0.canonical_support_output(multiset, output)] = 1\n            if su4_mask:\n                su4_survivors[stage0.canonical_support_output(multiset, output)] = 1\n            if stable_mask & ~su4_mask:\n                raise AssertionError("stable assignment absent from SU(4) sector")\n        if index % 40000 == 0:\n            print(\n                f"[geometry] supports={index:,}/{len(supports):,} "\n                f"stable_classes={len(stable_survivors):,} su4_classes={len(su4_survivors):,}",\n                flush=True,\n            )\n\n    gate("candidate support/output pair count", candidate_pairs == 895524, f"{candidate_pairs:,}")\n    gate("stable support/output classes reproduced", len(stable_survivors) == 439, str(len(stable_survivors)))\n    gate("stable support classes are an SU(4) subset",\n         set(stable_survivors).issubset(set(su4_survivors)),\n         f"stable={len(stable_survivors)} su4={len(su4_survivors)}")\n\n    def ordered_keys_from(survivors: dict[Any, int]) -> set[Any]:\n        keys = set()\n        for multiset, output in survivors:\n            for word in set(itertools.permutations(multiset)):\n                keys.add(stage0.canonical_ordered_transition(word, output))\n        return keys\n\n    stable_ordered_keys = ordered_keys_from(stable_survivors)\n    su4_ordered_keys = ordered_keys_from(su4_survivors)\n    gate("stable ordered-key candidates are an SU(4) subset",\n         stable_ordered_keys.issubset(su4_ordered_keys),\n         f"stable={len(stable_ordered_keys)} su4={len(su4_ordered_keys)}")\n\n    stable_ordered: dict[Any, int] = {}\n    su4_ordered: dict[Any, int] = {}\n    exceptional_ordered: dict[Any, int] = {}\n\n    for idx, (word, output) in enumerate(sorted(su4_ordered_keys), start=1):\n        stable_mask = mask_for(word, output, su4=False)\n        su4_mask = mask_for(word, output, su4=True)\n        exceptional_mask = su4_mask & ~stable_mask\n        if stable_mask:\n            stable_ordered[(word, output)] = stable_mask\n        if su4_mask:\n            su4_ordered[(word, output)] = su4_mask\n        if exceptional_mask:\n            exceptional_ordered[(word, output)] = exceptional_mask\n        if idx % 10000 == 0:\n            print(\n                f"[ordered] keys={idx:,}/{len(su4_ordered_keys):,} "\n                f"stable={len(stable_ordered):,} exceptional={len(exceptional_ordered):,}",\n                flush=True,\n            )\n\n    stable_assignments = sum(mask.bit_count() for mask in stable_ordered.values())\n    gate("stable ordered words reproduced", len(stable_ordered) == 4171, str(len(stable_ordered)))\n    gate("stable sign assignments reproduced", stable_assignments == 33500, f"{stable_assignments:,}")\n    gate("stable charge-conjugation orbits reproduced", stable_assignments // 2 == 16750,\n         f"{stable_assignments // 2:,}")\n\n    archive_payload = read_json_gz(words_path)\n    archive_records = archive_payload["words"]\n    gate("reference archive record count", len(archive_records) == 4171, str(len(archive_records)))\n    archive_map = {key_from_record(r): assignments_from_record(r) for r in archive_records}\n    computed_map = {\n        key: {signs_basis[i] for i in bit_indices(mask)}\n        for key, mask in stable_ordered.items()\n    }\n    gate("stable ordered-word key set exactly matches reference archive",\n         set(computed_map) == set(archive_map),\n         f"computed={len(computed_map)} archive={len(archive_map)}")\n    mismatch_keys = [k for k in archive_map if archive_map[k] != computed_map.get(k)]\n    gate("all stable assignment sets exactly match reference archive",\n         not mismatch_keys, f"mismatches={len(mismatch_keys)}")\n\n    exceptional_assignment_count = sum(mask.bit_count() for mask in exceptional_ordered.values())\n    gate("exceptional assignment count is even under charge conjugation",\n         exceptional_assignment_count % 2 == 0, str(exceptional_assignment_count))\n\n    new_word_keys = set(su4_ordered) - set(stable_ordered)\n    mixed_word_keys = set(exceptional_ordered) & set(stable_ordered)\n    pure_exceptional_keys = set(exceptional_ordered) - set(stable_ordered)\n    gate("new SU(4) words are exactly pure exceptional words",\n         new_word_keys == pure_exceptional_keys,\n         f"new={len(new_word_keys)} pure={len(pure_exceptional_keys)}")\n\n    archive_id = {key_from_record(r): r["ordered_id"] for r in archive_records}\n\n    final_family_hist: Counter[tuple[int, int]] = Counter()\n    final_charge_hist: Counter[int] = Counter()\n    exceptional_links_per_assignment: Counter[int] = Counter()\n    cut_det_hist: Counter[int] = Counter()\n    cut_det_family_hist: Counter[tuple[int, int, int]] = Counter()\n    assignments_with_resolvent_det = 0\n    assignments_final_only = 0\n    signature_occurrences: Counter[tuple[int, ...]] = Counter()\n    canonical_signature_occurrences: Counter[tuple[int, ...]] = Counter()\n    signature_cut_profiles: dict[tuple[int, ...], tuple[int, int, int]] = {}\n    bad: list[dict[str, Any]] = []\n    exceptional_records: list[dict[str, Any]] = []\n\n    for record_index, ((word, output), exceptional_mask) in enumerate(\n        sorted(exceptional_ordered.items()), start=1\n    ):\n        factors = (stage0.ROOT,) + tuple(word) + (output,)\n        links = sorted({\n            link\n            for plaquette in factors\n            for link, _incidence in stage1_local.boundary(plaquette)\n        })\n        assignment_records = []\n        for sign_index in bit_indices(exceptional_mask):\n            signs = signs_basis[sign_index]\n            local_records = []\n            has_resolvent_det = False\n            exceptional_link_count = 0\n            for link in links:\n                tokens = tuple(int(x) for x in stage1_local.factor_tokens(factors, signs, link))\n                nf = sum(t == 1 for t in tokens)\n                na = sum(t == -1 for t in tokens)\n                charge = nf - na\n                if charge == 0:\n                    continue\n                exceptional_link_count += 1\n                final_family_hist[(nf, na)] += 1\n                final_charge_hist[charge] += 1\n                signature_occurrences[tokens] += 1\n                canonical_signature_occurrences[signature_representative(tokens)] += 1\n                cut_charges = tuple(sum(tokens[:cut + 1]) for cut in (1, 2, 3))\n                signature_cut_profiles[tokens] = cut_charges\n                det_cuts = []\n                for cut_number, qcut in zip((1, 2, 3), cut_charges):\n                    if abs(qcut) == 4:\n                        prefix = tokens[:cut_number + 1]\n                        pnf = sum(t == 1 for t in prefix)\n                        pna = sum(t == -1 for t in prefix)\n                        det_cuts.append(cut_number)\n                        cut_det_hist[cut_number] += 1\n                        cut_det_family_hist[(cut_number, pnf, pna)] += 1\n                        has_resolvent_det = True\n                    elif abs(qcut) > 4:\n                        bad.append({\n                            "reason": "unexpected_prefix_charge",\n                            "tokens": tokens, "cut": cut_number, "charge": qcut,\n                        })\n                if charge not in (-4, 4):\n                    bad.append({\n                        "reason": "unexpected_final_charge",\n                        "tokens": tokens, "charge": charge,\n                    })\n                if (nf, na) not in {(4, 0), (0, 4), (5, 1), (1, 5)}:\n                    bad.append({\n                        "reason": "unexpected_final_family",\n                        "tokens": tokens, "family": (nf, na),\n                    })\n                local_records.append({\n                    "link": repr(link),\n                    "tokens": list(tokens),\n                    "family": [nf, na],\n                    "charge": charge,\n                    "cut_charges": list(cut_charges),\n                    "determinant_cuts": det_cuts,\n                })\n            if exceptional_link_count == 0:\n                bad.append({\n                    "reason": "exceptional_assignment_without_exceptional_link",\n                    "word": word, "output": output, "signs": signs,\n                })\n            exceptional_links_per_assignment[exceptional_link_count] += 1\n            if has_resolvent_det:\n                assignments_with_resolvent_det += 1\n            else:\n                assignments_final_only += 1\n            assignment_records.append({\n                "signs": list(signs),\n                "c_representative": list(c_representative(signs)),\n                "has_resolvent_determinant": has_resolvent_det,\n                "exceptional_link_count": exceptional_link_count,\n                "local_records": local_records,\n            })\n        exceptional_records.append({\n            "su4_exceptional_id": f"S4X4-{record_index:05d}",\n            "stable_ordered_id": archive_id.get((word, output)),\n            "root": list(stage0.ROOT),\n            "ordered_insertions": [list(p) for p in word],\n            "output": list(output),\n            "word_sector": (\n                "mixed_stable_and_exceptional"\n                if (word, output) in stable_ordered\n                else "exceptional_only"\n            ),\n            "exceptional_assignment_count": exceptional_mask.bit_count(),\n            "assignments": assignment_records,\n        })\n\n    gate("every exceptional assignment contains a nonzero SU(4) determinant-family link",\n         not bad, f"bad={len(bad)}")\n    gate("only final local charges +4 and -4 occur",\n         set(final_charge_hist).issubset({-4, 4}), str(dict(final_charge_hist)))\n    gate("only SU(4) exceptional final families occur",\n         set(final_family_hist).issubset({(4, 0), (0, 4), (5, 1), (1, 5)}),\n         str(dict(final_family_hist)))\n    gate("resolvent determinant prefixes are pure four-strand sectors",\n         all((pnf, pna) in {(4, 0), (0, 4)}\n             for _cut, pnf, pna in cut_det_family_hist),\n         str(dict(cut_det_family_hist)))\n    gate("maximum final local tensor degree remains six", max_local_degree <= 6, str(max_local_degree))\n    gate("all exceptional assignments classified",\n         assignments_with_resolvent_det + assignments_final_only == exceptional_assignment_count,\n         f"{assignments_with_resolvent_det}+{assignments_final_only}={exceptional_assignment_count}")\n\n    all_records = []\n    for idx, (key, su4_mask) in enumerate(sorted(su4_ordered.items()), start=1):\n        word, output = key\n        stable_mask = stable_ordered.get(key, 0)\n        exceptional_mask = exceptional_ordered.get(key, 0)\n        all_records.append({\n            "su4_ordered_id": f"S4N4-{idx:05d}",\n            "stable_ordered_id": archive_id.get(key),\n            "root": list(stage0.ROOT),\n            "ordered_insertions": [list(p) for p in word],\n            "output": list(output),\n            "su4_assignment_count": su4_mask.bit_count(),\n            "stable_assignment_count": stable_mask.bit_count(),\n            "exceptional_assignment_count": exceptional_mask.bit_count(),\n            "stable_assignments": [list(signs_basis[i]) for i in bit_indices(stable_mask)],\n            "exceptional_assignments": [list(signs_basis[i]) for i in bit_indices(exceptional_mask)],\n            "word_sector": (\n                "mixed_stable_and_exceptional" if stable_mask and exceptional_mask\n                else "stable_only" if stable_mask\n                else "exceptional_only"\n            ),\n        })\n\n    all_path = OUT / "y4_su4_ordered_words.json.gz"\n    exceptional_path = OUT / "y4_su4_exceptional_only_words.json.gz"\n    catalog_path = OUT / "y4_su4_local_signature_catalog.json"\n\n    all_sha = write_json_gz(all_path, {\n        "meta": {\n            "version": VERSION, "rank": N,\n            "criterion": "local charge is 0 modulo 4",\n            "stable_reference": str(words_path),\n            "stable_reference_sha256": sha256(words_path),\n        },\n        "counts": {\n            "ordered_words": len(all_records),\n            "stable_words": len(stable_ordered),\n            "exceptional_bearing_words": len(exceptional_ordered),\n            "new_exceptional_only_words": len(new_word_keys),\n            "mixed_words": len(mixed_word_keys),\n            "stable_assignments": stable_assignments,\n            "exceptional_assignments": exceptional_assignment_count,\n        },\n        "words": all_records,\n    })\n    exceptional_sha = write_json_gz(exceptional_path, {\n        "meta": {\n            "version": VERSION, "rank": N,\n            "scope": "assignments absent from the N>=7 exact-balance corpus",\n            "criterion": "at least one final local charge is +4 or -4",\n        },\n        "counts": {\n            "exceptional_bearing_words": len(exceptional_records),\n            "new_exceptional_only_words": len(new_word_keys),\n            "mixed_words": len(mixed_word_keys),\n            "exceptional_assignments": exceptional_assignment_count,\n            "exceptional_charge_conjugation_orbits": exceptional_assignment_count // 2,\n            "assignments_with_resolvent_determinant": assignments_with_resolvent_det,\n            "assignments_final_only": assignments_final_only,\n        },\n        "words": exceptional_records,\n    })\n\n    catalog_rows = []\n    for sig, count in sorted(signature_occurrences.items()):\n        nf = sig.count(1)\n        na = sig.count(-1)\n        catalog_rows.append({\n            "signature": list(sig),\n            "canonical_under_C": list(signature_representative(sig)),\n            "occurrences": count,\n            "family": [nf, na],\n            "final_charge": nf - na,\n            "cut_charges": list(signature_cut_profiles[sig]),\n            "determinant_cuts": [\n                cut for cut, q in zip((1, 2, 3), signature_cut_profiles[sig]) if abs(q) == 4\n            ],\n        })\n    catalog_sha = write_json(catalog_path, {\n        "meta": {"version": VERSION, "rank": N},\n        "counts": {\n            "oriented_signatures": len(signature_occurrences),\n            "C_canonical_signatures": len(canonical_signature_occurrences),\n            "total_occurrences": sum(signature_occurrences.values()),\n        },\n        "final_family_histogram": {\n            f"{a},{b}": v for (a, b), v in sorted(final_family_hist.items())\n        },\n        "cut_determinant_histogram": {str(k): v for k, v in sorted(cut_det_hist.items())},\n        "cut_determinant_family_histogram": {\n            f"cut={cut};family=({a},{b})": v\n            for (cut, a, b), v in sorted(cut_det_family_hist.items())\n        },\n        "signatures": catalog_rows,\n    })\n\n    next_stage = (\n        "A finite-rank SU(4) local library is required because determinant channels occur "\n        "inside resolvent cuts. Treat every pure four-strand prefix as the SU(4) singlet "\n        "Lambda^4 V with C2=0, and implement final epsilon/delta Haar tensors for the "\n        "(4,0),(0,4),(5,1),(1,5) families. Preserve the stable 4,171-word contraction unchanged."\n        if assignments_with_resolvent_det\n        else\n        "No determinant channel occurs in a resolvent cut; contract only the final exceptional Haar nodes."\n    )\n\n    summary = {\n        "version": VERSION,\n        "status": "PASS",\n        "inputs": {\n            "stage1_source": str(st_path),\n            "stage1_source_sha256": sha256(st_path),\n            "stable_words": str(words_path),\n            "stable_words_sha256": sha256(words_path),\n            "complete_source": str(complete_source),\n            "supports": str(support_path),\n            "supports_sha256": sha256(support_path),\n            "extraction": extraction,\n        },\n        "stable_regression": {\n            "support_output_classes": len(stable_survivors),\n            "ordered_words": len(stable_ordered),\n            "sign_assignments": stable_assignments,\n            "charge_conjugation_orbits": stable_assignments // 2,\n            "archive_key_set_exact": True,\n            "archive_assignment_sets_exact": True,\n        },\n        "su4": {\n            "support_output_classes": len(su4_survivors),\n            "ordered_words": len(su4_ordered),\n            "total_assignments": sum(mask.bit_count() for mask in su4_ordered.values()),\n            "exceptional_bearing_words": len(exceptional_ordered),\n            "new_exceptional_only_words": len(new_word_keys),\n            "mixed_words": len(mixed_word_keys),\n            "exceptional_assignments": exceptional_assignment_count,\n            "exceptional_charge_conjugation_orbits": exceptional_assignment_count // 2,\n            "assignments_with_resolvent_determinant": assignments_with_resolvent_det,\n            "assignments_final_only": assignments_final_only,\n            "final_family_histogram": {\n                f"({a},{b})": v for (a, b), v in sorted(final_family_hist.items())\n            },\n            "final_charge_histogram": {str(k): v for k, v in sorted(final_charge_hist.items())},\n            "exceptional_links_per_assignment": {\n                str(k): v for k, v in sorted(exceptional_links_per_assignment.items())\n            },\n            "cut_determinant_histogram": {str(k): v for k, v in sorted(cut_det_hist.items())},\n            "oriented_local_signatures": len(signature_occurrences),\n            "C_canonical_local_signatures": len(canonical_signature_occurrences),\n            "max_local_degree": max_local_degree,\n        },\n        "outputs": {\n            "all_words": str(all_path), "all_words_sha256": all_sha,\n            "exceptional_words": str(exceptional_path), "exceptional_words_sha256": exceptional_sha,\n            "signature_catalog": str(catalog_path), "signature_catalog_sha256": catalog_sha,\n        },\n        "next_stage": next_stage,\n        "elapsed_seconds": time.time() - started,\n    }\n\n    json_path = OUT / "SU4_EXCEPTIONAL_ENUMERATOR_V1.json"\n    json_sha = write_json(json_path, summary)\n\n    md = f"""# SU(4) exceptional-rank O(y^4) corpus\n\n**Status:** PASS  \n**Version:** `{VERSION}`\n\n## Stable-rank regression\n\n- support/output classes: **{len(stable_survivors):,}**\n- ordered words: **{len(stable_ordered):,}**\n- sign assignments: **{stable_assignments:,}**\n- charge-conjugation orbits: **{stable_assignments // 2:,}**\n- archive keys and assignment sets: **exact match**\n\n## SU(4) extension\n\n- support/output classes: **{len(su4_survivors):,}**\n- ordered words: **{len(su4_ordered):,}**\n- exceptional-bearing words: **{len(exceptional_ordered):,}**\n- new exceptional-only words: **{len(new_word_keys):,}**\n- mixed stable/exceptional words: **{len(mixed_word_keys):,}**\n- exceptional assignments: **{exceptional_assignment_count:,}**\n- exceptional C-orbits: **{exceptional_assignment_count // 2:,}**\n\nFinal exceptional family histogram:\n\n```text\n{dict(sorted(final_family_hist.items()))}\n```\n\nAssignments with a determinant channel in at least one resolvent cut:\n\n```text\n{assignments_with_resolvent_det:,}\n```\n\nAssignments whose determinant structure occurs only in the final Haar integral:\n\n```text\n{assignments_final_only:,}\n```\n\nCut histogram for pure `Lambda^4 V` determinant channels:\n\n```text\n{dict(sorted(cut_det_hist.items()))}\n```\n\n## Consequence\n\n{next_stage}\n\n## Outputs\n\n- `{all_path.name}`\n- `{exceptional_path.name}`\n- `{catalog_path.name}`\n- `{json_path.name}`\n"""\n    md_path = OUT / "SU4_EXCEPTIONAL_ENUMERATOR_V1.md"\n    md_path.write_text(md, encoding="utf-8")\n    md_sha = sha256(md_path)\n\n    bundle = BASE / "SU4_EXCEPTIONAL_ENUMERATOR_V1_BUNDLE.zip"\n    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as zf:\n        for p in (json_path, md_path, all_path, exceptional_path, catalog_path):\n            zf.write(p, arcname=p.name)\n        source_path = Path(globals().get("__file__", ""))\n        if source_path.is_file():\n            zf.write(source_path, arcname=source_path.name)\n\n    print("\\n" + "=" * 116)\n    print("SU(4) EXCEPTIONAL ENUMERATOR STATUS: PASS")\n    print("=" * 116)\n    print(f"stable words                         : {len(stable_ordered):,}")\n    print(f"SU(4) words                          : {len(su4_ordered):,}")\n    print(f"exceptional-bearing words            : {len(exceptional_ordered):,}")\n    print(f"new exceptional-only words           : {len(new_word_keys):,}")\n    print(f"mixed words                          : {len(mixed_word_keys):,}")\n    print(f"exceptional assignments              : {exceptional_assignment_count:,}")\n    print(f"exceptional C-orbits                 : {exceptional_assignment_count // 2:,}")\n    print(f"assignments with resolvent determinant: {assignments_with_resolvent_det:,}")\n    print(f"final-only exceptional assignments   : {assignments_final_only:,}")\n    print("final family histogram               :", dict(sorted(final_family_hist.items())))\n    print("cut determinant histogram            :", dict(sorted(cut_det_hist.items())))\n    print("JSON:", json_path, json_sha)\n    print("MD:  ", md_path, md_sha)\n    print("ALL: ", all_path, all_sha)\n    print("EXC: ", exceptional_path, exceptional_sha)\n    print("SIG: ", catalog_path, catalog_sha)\n    print("ZIP: ", bundle, sha256(bundle))\n    print("=" * 116)\n\n\nif __name__ == "__main__":\n    main()\n'
ALG_SOURCE = '#!/usr/bin/env python3\n"""\nSU(4) exceptional local Haar/Casimir algebra certificate.\n\nConsumes the output of y4_su4_exceptional_enumerator_v1.py and constructs,\nin exact arithmetic, the finite-rank local library needed by the global O(y^4)\ncontraction.\n\nFor every oriented exceptional signature, this script:\n\n  * builds an explicit invariant basis:\n      (4,0)/(0,4): epsilon tensor;\n      (5,1)/(1,5): five delta-epsilon tensors of rank four;\n  * removes the unique five-index antisymmetry relation;\n  * constructs the exact Gram matrix;\n  * builds the three nested prefix Casimirs by Fierz identities;\n  * verifies exact closure, G-self-adjointness, and mutual commutativity;\n  * performs the exact joint eigenspace decomposition;\n  * exports each channel as\n        sum_ab K_ab T_a(row) T_b(col)\n    together with its (C2_cut1,C2_cut2,C2_cut3) history;\n  * proves the channel projectors are orthogonal, idempotent, and complete.\n\nInput, preferred:\n  SU4_EXCEPTIONAL_ENUMERATOR_V1_BUNDLE.zip\n\nFallback:\n  y4_su4_local_signature_catalog.json\n\nOutputs:\n  SU4_LOCAL_ALGEBRA_V1/SU4_LOCAL_ALGEBRA_V1.json\n  SU4_LOCAL_ALGEBRA_V1/SU4_LOCAL_ALGEBRA_V1.md\n  SU4_LOCAL_ALGEBRA_V1/y4_su4_exceptional_local_library.json\n  SU4_LOCAL_ALGEBRA_V1_BUNDLE.zip\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport itertools\nimport json\nimport os\nimport re\nimport sys\nimport time\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom fractions import Fraction\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport sympy as sp\n\nVERSION = "2026-06-14-su4-local-algebra-v1"\nN = 4\nCUTS = (1, 2, 3)\nBASE = Path("/content") if Path("/content").exists() else Path("/mnt/data")\nOUT = BASE / "SU4_LOCAL_ALGEBRA_V1"\nEXTRACT = OUT / "extracted"\nOUT.mkdir(parents=True, exist_ok=True)\nEXTRACT.mkdir(parents=True, exist_ok=True)\n\nENUM_BUNDLE_GLOB = "SU4_EXCEPTIONAL_ENUMERATOR_V1_BUNDLE*.zip"\nCATALOG_NAME = "y4_su4_local_signature_catalog.json"\n\n\ndef gate(name: str, cond: bool, detail: str = "") -> None:\n    status = "PASS" if cond else "FAIL"\n    print(f"{status:4s} {name:92s} {detail}")\n    if not cond:\n        raise AssertionError(f"{name}: {detail}")\n\n\ndef sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for block in iter(lambda: f.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef safe_extract(zpath: Path, dest: Path) -> list[Path]:\n    dest.mkdir(parents=True, exist_ok=True)\n    root = dest.resolve()\n    with zipfile.ZipFile(zpath) as zf:\n        for info in zf.infolist():\n            target = (dest / info.filename).resolve()\n            if target != root and not str(target).startswith(str(root) + os.sep):\n                raise ValueError(f"unsafe ZIP member: {info.filename}")\n        zf.extractall(dest)\n        return [dest / x.filename for x in zf.infolist() if not x.is_dir()]\n\n\ndef recursive_extract(archives: Iterable[Path], max_depth: int = 3) -> list[dict[str, Any]]:\n    queue = [(Path(p), 0) for p in archives]\n    seen: set[str] = set()\n    records: list[dict[str, Any]] = []\n    while queue:\n        zp, depth = queue.pop(0)\n        if depth > max_depth or not zp.is_file():\n            continue\n        h = sha256(zp)\n        if h in seen:\n            continue\n        seen.add(h)\n        label = re.sub(r"[^A-Za-z0-9_.-]+", "_", zp.stem)[:90]\n        dest = EXTRACT / f"d{depth}_{label}_{h[:10]}"\n        try:\n            files = safe_extract(zp, dest)\n            records.append({\n                "archive": str(zp), "sha256": h, "depth": depth,\n                "destination": str(dest), "file_count": len(files), "status": "ok",\n            })\n            for p in files:\n                if p.suffix.lower() == ".zip":\n                    queue.append((p, depth + 1))\n        except Exception as exc:\n            records.append({\n                "archive": str(zp), "sha256": h, "depth": depth,\n                "status": "error", "error": repr(exc),\n            })\n    return records\n\n\ndef find_all(name: str, roots: Iterable[Path]) -> list[Path]:\n    found: dict[str, Path] = {}\n    for root in roots:\n        if not root.exists():\n            continue\n        for p in root.rglob(name):\n            if p.is_file():\n                found[sha256(p)] = p\n    return sorted(found.values(), key=lambda p: str(p))\n\n\ndef find_glob(pattern: str, roots: Iterable[Path]) -> list[Path]:\n    found: dict[str, Path] = {}\n    for root in roots:\n        if not root.exists():\n            continue\n        for p in root.rglob(pattern):\n            if p.is_file():\n                found[sha256(p)] = p\n    return sorted(found.values(), key=lambda p: str(p))\n\n\ndef upload_if_needed() -> list[Path]:\n    if find_all(CATALOG_NAME, [BASE]) or find_glob(ENUM_BUNDLE_GLOB, [BASE]):\n        return []\n    if not Path("/content").exists():\n        return []\n    try:\n        from google.colab import files as colab_files  # type: ignore\n    except Exception:\n        return []\n    print("\\nUPLOAD REQUIRED — select:")\n    print("  SU4_EXCEPTIONAL_ENUMERATOR_V1_BUNDLE.zip")\n    print("or:")\n    print("  y4_su4_local_signature_catalog.json")\n    uploaded = colab_files.upload()\n    saved: list[Path] = []\n    for name, data in uploaded.items():\n        target = Path("/content") / Path(name).name\n        target.write_bytes(data)\n        saved.append(target)\n        print(f"saved {len(data):,} bytes -> {target}")\n    return saved\n\n\ndef parity(p: tuple[int, ...]) -> int:\n    inversions = sum(\n        p[i] > p[j]\n        for i in range(len(p))\n        for j in range(i + 1, len(p))\n    )\n    return -1 if inversions & 1 else 1\n\n\ndef flat_index(values: tuple[int, ...]) -> int:\n    out = 0\n    for value in values:\n        out = N * out + value\n    return out\n\n\ndef unflat_index(index: int, length: int) -> tuple[int, ...]:\n    values = [0] * length\n    for i in range(length - 1, -1, -1):\n        values[i] = index % N\n        index //= N\n    return tuple(values)\n\n\nVector = dict[int, Fraction]\n\n\ndef clean(v: Vector) -> Vector:\n    return {k: x for k, x in v.items() if x}\n\n\ndef add_scaled(out: Vector, v: Vector, coefficient: Fraction) -> None:\n    if not coefficient:\n        return\n    for key, value in v.items():\n        out[key] = out.get(key, Fraction(0)) + coefficient * value\n\n\ndef dot(a: Vector, b: Vector) -> Fraction:\n    if len(a) > len(b):\n        a, b = b, a\n    return sum(value * b.get(key, 0) for key, value in a.items())\n\n\ndef epsilon_vector(length: int, positions: list[int]) -> Vector:\n    out: Vector = {}\n    for permutation in itertools.permutations(range(N)):\n        values = [0] * length\n        for position, color in zip(positions, permutation):\n            values[position] = color\n        out[flat_index(tuple(values))] = Fraction(parity(permutation))\n    return out\n\n\ndef delta_epsilon_vector(\n    tokens: tuple[int, ...],\n    chosen_excess_position: int,\n) -> Vector:\n    plus = [i for i, token in enumerate(tokens) if token == 1]\n    minus = [i for i, token in enumerate(tokens) if token == -1]\n    if len(plus) == 5 and len(minus) == 1:\n        excess = plus\n        paired_position = minus[0]\n    elif len(minus) == 5 and len(plus) == 1:\n        excess = minus\n        paired_position = plus[0]\n    else:\n        raise ValueError((tokens, chosen_excess_position))\n\n    remaining = [i for i in excess if i != chosen_excess_position]\n    out: Vector = {}\n    for paired_color in range(N):\n        for permutation in itertools.permutations(range(N)):\n            values = [0] * len(tokens)\n            values[chosen_excess_position] = paired_color\n            values[paired_position] = paired_color\n            for position, color in zip(remaining, permutation):\n                values[position] = color\n            key = flat_index(tuple(values))\n            out[key] = out.get(key, Fraction(0)) + Fraction(parity(permutation))\n    return clean(out)\n\n\ndef swap_operator(v: Vector, length: int, a: int, b: int) -> Vector:\n    out: Vector = {}\n    for key, value in v.items():\n        values = list(unflat_index(key, length))\n        values[a], values[b] = values[b], values[a]\n        new_key = flat_index(tuple(values))\n        out[new_key] = out.get(new_key, Fraction(0)) + value\n    return clean(out)\n\n\ndef contraction_operator(v: Vector, length: int, a: int, b: int) -> Vector:\n    """\n    K on V tensor V*: K|i,j> = delta_ij sum_c |c,c>.\n    """\n    grouped: dict[tuple[int, ...], Fraction] = defaultdict(Fraction)\n    for key, value in v.items():\n        values = unflat_index(key, length)\n        if values[a] == values[b]:\n            remaining = tuple(\n                values[i] for i in range(length) if i not in (a, b)\n            )\n            grouped[remaining] += value\n\n    out: Vector = {}\n    for remaining, coefficient in grouped.items():\n        for color in range(N):\n            values = []\n            iterator = iter(remaining)\n            for i in range(length):\n                values.append(color if i in (a, b) else next(iterator))\n            key = flat_index(tuple(values))\n            out[key] = out.get(key, Fraction(0)) + coefficient\n    return clean(out)\n\n\ndef casimir_apply(\n    v: Vector,\n    tokens: tuple[int, ...],\n    event_positions: tuple[int, ...],\n    cut: int,\n) -> Vector:\n    prefix = [\n        local_position\n        for local_position, event_position in enumerate(event_positions)\n        if event_position <= cut\n    ]\n    out: Vector = {}\n    c_f = Fraction(N * N - 1, 2 * N)\n    add_scaled(out, v, len(prefix) * c_f)\n\n    for offset, a in enumerate(prefix):\n        for b in prefix[offset + 1:]:\n            if tokens[a] == tokens[b]:\n                # 2 T_a.T_b = P_ab - I/N.\n                add_scaled(out, swap_operator(v, len(tokens), a, b), Fraction(1))\n                add_scaled(out, v, Fraction(-1, N))\n            else:\n                # 2 T_a.T_bbar = -K_ab + I/N.\n                add_scaled(out, contraction_operator(v, len(tokens), a, b), Fraction(-1))\n                add_scaled(out, v, Fraction(1, N))\n    return clean(out)\n\n\ndef to_sympy(value: Fraction) -> sp.Rational:\n    return sp.Rational(value.numerator, value.denominator)\n\n\ndef matrix_to_strings(matrix: sp.Matrix) -> list[list[str]]:\n    return [[str(sp.factor(matrix[i, j])) for j in range(matrix.cols)]\n            for i in range(matrix.rows)]\n\n\ndef vector_linear_combination(\n    basis: list[Vector],\n    coefficients: list[Fraction],\n) -> Vector:\n    out: Vector = {}\n    for vector, coefficient in zip(basis, coefficients):\n        add_scaled(out, vector, coefficient)\n    return clean(out)\n\n\ndef restricted_matrix(M: sp.Matrix, S: sp.Matrix) -> sp.Matrix:\n    """Return X with M S = S X, using an exact invertible row minor."""\n    rank = S.cols\n    pivot_rows = list(S.T.rref()[1])\n    minor = S.extract(pivot_rows, range(rank))\n    gate("joint-decomposition subspace minor is nonsingular",\n         minor.det() != 0, f"rank={rank}")\n    rhs = (M * S).extract(pivot_rows, range(rank))\n    X = sp.simplify(minor.inv() * rhs)\n    gate("restricted Casimir preserves current joint subspace",\n         M * S == S * X, "")\n    return X\n\n\ndef joint_eigenspaces(matrices: list[sp.Matrix]) -> list[dict[str, Any]]:\n    dimension = matrices[0].rows\n    spaces = [{"basis": sp.eye(dimension), "eigenvalues": []}]\n    for M in matrices:\n        refined = []\n        for record in spaces:\n            S = record["basis"]\n            X = restricted_matrix(M, S)\n            for eigenvalue, multiplicity, eigenvectors in X.eigenvects():\n                V = sp.Matrix.hstack(*eigenvectors)\n                gate("eigenspace dimension matches algebraic multiplicity",\n                     V.cols == multiplicity,\n                     f"lambda={eigenvalue} cols={V.cols} multiplicity={multiplicity}")\n                refined.append({\n                    "basis": sp.simplify(S * V),\n                    "eigenvalues": record["eigenvalues"] + [sp.factor(eigenvalue)],\n                })\n        spaces = refined\n    return spaces\n\n\ndef candidate_basis(signature: tuple[int, ...]) -> tuple[\n    tuple[int, ...], tuple[int, ...], list[dict[str, Any]], list[Vector]\n]:\n    event_positions = tuple(i for i, token in enumerate(signature) if token)\n    tokens = tuple(signature[i] for i in event_positions)\n    plus = tokens.count(1)\n    minus = tokens.count(-1)\n\n    descriptors: list[dict[str, Any]] = []\n    vectors: list[Vector] = []\n\n    if (plus, minus) in {(4, 0), (0, 4)}:\n        descriptors.append({\n            "type": "epsilon",\n            "event_positions": list(event_positions),\n            "orientation": "fundamental" if plus == 4 else "antifundamental",\n        })\n        vectors.append(epsilon_vector(len(tokens), list(range(len(tokens)))))\n    elif (plus, minus) in {(5, 1), (1, 5)}:\n        excess_token = 1 if plus == 5 else -1\n        excess_positions = [\n            i for i, token in enumerate(tokens) if token == excess_token\n        ]\n        paired_position = next(\n            i for i, token in enumerate(tokens) if token == -excess_token\n        )\n        for chosen in excess_positions:\n            descriptors.append({\n                "type": "delta_epsilon",\n                "chosen_excess_local_position": chosen,\n                "chosen_excess_event": event_positions[chosen],\n                "paired_local_position": paired_position,\n                "paired_event": event_positions[paired_position],\n                "epsilon_events": [\n                    event_positions[i] for i in excess_positions if i != chosen\n                ],\n                "excess_orientation": (\n                    "fundamental" if excess_token == 1 else "antifundamental"\n                ),\n            })\n            vectors.append(delta_epsilon_vector(tokens, chosen))\n    else:\n        raise ValueError((signature, plus, minus))\n\n    return event_positions, tokens, descriptors, vectors\n\n\ndef analyze_signature(row: dict[str, Any]) -> dict[str, Any]:\n    signature = tuple(int(x) for x in row["signature"])\n    event_positions, tokens, descriptors, candidates = candidate_basis(signature)\n    family = (tokens.count(1), tokens.count(-1))\n    expected_rank = 1 if family in {(4, 0), (0, 4)} else 4\n\n    candidate_gram = sp.Matrix([\n        [to_sympy(dot(a, b)) for b in candidates]\n        for a in candidates\n    ])\n    rank = candidate_gram.rank()\n    gate("exceptional invariant-space rank",\n         rank == expected_rank,\n         f"signature={signature} family={family} rank={rank}")\n\n    pivots = list(candidate_gram.rref()[1])\n    gate("independent invariant basis has expected size",\n         len(pivots) == expected_rank,\n         f"signature={signature} pivots={pivots}")\n\n    basis = [candidates[i] for i in pivots]\n    basis_descriptors = [descriptors[i] for i in pivots]\n    G = candidate_gram.extract(pivots, pivots)\n    gate("independent invariant Gram matrix is positive definite",\n         all(x > 0 for x in G.cholesky().diagonal()),\n         f"signature={signature}")\n\n    cut_matrices: list[sp.Matrix] = []\n    cut_eigenvalue_histograms: list[dict[str, int]] = []\n\n    for cut in CUTS:\n        operated = [\n            casimir_apply(vector, tokens, event_positions, cut)\n            for vector in basis\n        ]\n        H = sp.Matrix([\n            [to_sympy(dot(left, right)) for right in operated]\n            for left in basis\n        ])\n        M = sp.simplify(G.inv() * H)\n\n        # Exact closure in the invariant basis.\n        for column, operated_vector in enumerate(operated):\n            coefficients = [\n                Fraction(int(sp.numer(M[i, column])), int(sp.denom(M[i, column])))\n                for i in range(M.rows)\n            ]\n            reconstructed = vector_linear_combination(basis, coefficients)\n            gate("prefix Casimir closes on invariant basis",\n                 reconstructed == operated_vector,\n                 f"signature={signature} cut={cut} column={column}")\n\n        gate("prefix Casimir is self-adjoint in the invariant Gram metric",\n             G * M == M.T * G,\n             f"signature={signature} cut={cut}")\n        eigenvalues = M.eigenvals()\n        gate("prefix Casimir spectrum is rational and nonnegative",\n             all(value.is_Rational and value >= 0 for value in eigenvalues),\n             f"signature={signature} cut={cut} eigenvalues={eigenvalues}")\n\n        cut_matrices.append(M)\n        cut_eigenvalue_histograms.append({\n            str(sp.factor(value)): int(multiplicity)\n            for value, multiplicity in sorted(\n                eigenvalues.items(), key=lambda item: sp.default_sort_key(item[0])\n            )\n        })\n\n    for i in range(len(cut_matrices)):\n        for j in range(i + 1, len(cut_matrices)):\n            gate("nested prefix Casimirs commute exactly",\n                 cut_matrices[i] * cut_matrices[j]\n                 == cut_matrices[j] * cut_matrices[i],\n                 f"signature={signature} cuts={CUTS[i]},{CUTS[j]}")\n\n    spaces = joint_eigenspaces(cut_matrices)\n    gate("joint Casimir eigenspaces span invariant space",\n         sum(record["basis"].cols for record in spaces) == expected_rank,\n         f"signature={signature}")\n\n    channels = []\n    kernels = []\n    for channel_index, space in enumerate(spaces, start=1):\n        S = space["basis"]\n        Gs = sp.simplify(S.T * G * S)\n        K = sp.simplify(S * Gs.inv() * S.T)\n        multiplicity = S.cols\n\n        gate("joint channel projector is idempotent",\n             sp.simplify(K * G * K - K) == sp.zeros(K.rows),\n             f"signature={signature} channel={channel_index}")\n        gate("joint channel trace equals multiplicity",\n             sp.simplify(sp.trace(K * G)) == multiplicity,\n             f"signature={signature} channel={channel_index}")\n\n        kernels.append(K)\n        channels.append({\n            "channel_id": f"C{channel_index:02d}",\n            "casimir_history": [str(sp.factor(x)) for x in space["eigenvalues"]],\n            "multiplicity": multiplicity,\n            "basis_coefficient_subspace": matrix_to_strings(S),\n            "kernel_in_independent_invariant_basis": matrix_to_strings(K),\n            "kernel_nonzero_entries": sum(\n                K[i, j] != 0 for i in range(K.rows) for j in range(K.cols)\n            ),\n        })\n\n    for i in range(len(kernels)):\n        for j in range(i + 1, len(kernels)):\n            gate("distinct joint channel projectors are Gram-orthogonal",\n                 sp.simplify(kernels[i] * G * kernels[j])\n                 == sp.zeros(expected_rank),\n                 f"signature={signature} channels={i+1},{j+1}")\n\n    total_kernel = sum(kernels, sp.zeros(expected_rank))\n    gate("joint channel projectors resolve the full Haar projector",\n         sp.simplify(total_kernel - G.inv()) == sp.zeros(expected_rank),\n         f"signature={signature}")\n    gate("full Haar projector has trace equal to invariant dimension",\n         sp.simplify(sp.trace(G.inv() * G)) == expected_rank,\n         f"signature={signature}")\n\n    audit_det_cuts = tuple(int(x) for x in row.get("determinant_cuts", []))\n    zero_cut3_channels = sum(\n        channel["multiplicity"]\n        for channel in channels\n        if channel["casimir_history"][2] == "0"\n    )\n    if 3 in audit_det_cuts:\n        gate("enumerator determinant-cut flag has a C2=0 cut-3 channel",\n             zero_cut3_channels >= 1,\n             f"signature={signature} zero_mult={zero_cut3_channels}")\n    else:\n        gate("no unflagged C2=0 cut-3 determinant channel",\n             zero_cut3_channels == 0,\n             f"signature={signature} zero_mult={zero_cut3_channels}")\n\n    # For a pure four-strand final invariant the prefix channel is exterior power.\n    if family in {(4, 0), (0, 4)}:\n        expected = []\n        for cut in CUTS:\n            p = sum(event <= cut for event in event_positions)\n            expected.append(Fraction(p * (N - p) * (N + 1), 2 * N))\n        actual = tuple(\n            Fraction(int(sp.numer(cut_matrices[i][0, 0])),\n                     int(sp.denom(cut_matrices[i][0, 0])))\n            for i in range(3)\n        )\n        gate("pure determinant signature follows exterior-power Casimirs",\n             actual == tuple(expected),\n             f"signature={signature} actual={actual} expected={tuple(expected)}")\n\n    return {\n        "signature": list(signature),\n        "canonical_under_C": row.get("canonical_under_C"),\n        "occurrences": int(row.get("occurrences", 0)),\n        "family": list(family),\n        "final_charge": family[0] - family[1],\n        "event_positions": list(event_positions),\n        "active_tokens": list(tokens),\n        "audit_cut_charges": row.get("cut_charges", []),\n        "audit_determinant_cuts": list(audit_det_cuts),\n        "candidate_basis_size": len(candidates),\n        "invariant_dimension": expected_rank,\n        "candidate_gram": matrix_to_strings(candidate_gram),\n        "independent_candidate_indices": pivots,\n        "independent_basis_descriptors": basis_descriptors,\n        "independent_gram": matrix_to_strings(G),\n        "cut_casimir_matrices": {\n            str(cut): matrix_to_strings(matrix)\n            for cut, matrix in zip(CUTS, cut_matrices)\n        },\n        "cut_spectra": {\n            str(cut): spectrum\n            for cut, spectrum in zip(CUTS, cut_eigenvalue_histograms)\n        },\n        "channels": channels,\n        "cut3_determinant_channel_multiplicity": zero_cut3_channels,\n    }\n\n\ndef main() -> None:\n    started = time.time()\n    print("=" * 116)\n    print("SU(4) EXCEPTIONAL LOCAL HAAR + JOINT CASIMIR ALGEBRA")\n    print("=" * 116)\n    print("version :", VERSION)\n    print("output  :", OUT)\n    print("hardware: CPU exact sparse tensors and rational linear algebra")\n\n    uploaded = upload_if_needed()\n    archives = find_glob(ENUM_BUNDLE_GLOB, [BASE])\n    archives += [p for p in uploaded if p.suffix.lower() == ".zip"]\n    extraction = recursive_extract(archives)\n    roots = [BASE, EXTRACT]\n\n    catalog_paths = find_all(CATALOG_NAME, roots)\n    gate("SU(4) local-signature catalog located", bool(catalog_paths), str(catalog_paths[:3]))\n    catalog_path = catalog_paths[0]\n    catalog = json.loads(catalog_path.read_text(encoding="utf-8"))\n    rows = catalog["signatures"]\n\n    gate("catalog contains exceptional signatures", bool(rows), str(len(rows)))\n    gate("catalog families are exactly the SU(4) exceptional families",\n         {\n             tuple(int(x) for x in row["family"])\n             for row in rows\n         }.issubset({(4, 0), (0, 4), (5, 1), (1, 5)}),\n         "")\n\n    library_records = []\n    family_signature_hist = Counter()\n    family_occurrence_hist = Counter()\n    invariant_dimension_hist = Counter()\n    channel_history_hist = Counter()\n    total_channel_kernel_terms = 0\n    determinant_cut3_signature_count = 0\n\n    for index, row in enumerate(rows, start=1):\n        record = analyze_signature(row)\n        record["local_library_id"] = f"S4L-{index:04d}"\n        library_records.append(record)\n\n        family = tuple(record["family"])\n        family_signature_hist[family] += 1\n        family_occurrence_hist[family] += record["occurrences"]\n        invariant_dimension_hist[record["invariant_dimension"]] += 1\n        if record["cut3_determinant_channel_multiplicity"]:\n            determinant_cut3_signature_count += 1\n        for channel in record["channels"]:\n            history = tuple(channel["casimir_history"])\n            channel_history_hist[history] += record["occurrences"] * channel["multiplicity"]\n            total_channel_kernel_terms += channel["kernel_nonzero_entries"]\n\n        if index % 10 == 0 or index == len(rows):\n            print(\n                f"[local algebra] signatures={index}/{len(rows)} "\n                f"channels={sum(len(r[\'channels\']) for r in library_records)}",\n                flush=True,\n            )\n\n    # Charge-conjugation consistency.\n    record_by_signature = {\n        tuple(record["signature"]): record for record in library_records\n    }\n    for signature, record in record_by_signature.items():\n        conjugate = tuple(-x for x in signature)\n        if conjugate not in record_by_signature:\n            continue\n        other = record_by_signature[conjugate]\n        gate("charge-conjugate signatures have equal invariant dimension",\n             record["invariant_dimension"] == other["invariant_dimension"],\n             f"{signature}")\n        gate("charge-conjugate signatures have equal cut spectra",\n             record["cut_spectra"] == other["cut_spectra"],\n             f"{signature}")\n        histories = sorted(\n            (tuple(c["casimir_history"]), c["multiplicity"])\n            for c in record["channels"]\n        )\n        other_histories = sorted(\n            (tuple(c["casimir_history"]), c["multiplicity"])\n            for c in other["channels"]\n        )\n        gate("charge-conjugate signatures have equal joint Casimir histories",\n             histories == other_histories,\n             f"{signature}")\n\n    output_library = {\n        "meta": {\n            "version": VERSION,\n            "rank": N,\n            "normalization": "Tr(T^A T^B)=1/2 delta^{AB}",\n            "local_tensor_formula": (\n                "For each channel: P_channel(row,col)="\n                "sum_ab K_ab T_a(row)T_b(col), using independent_basis_descriptors."\n            ),\n            "source_catalog": str(catalog_path),\n            "source_catalog_sha256": sha256(catalog_path),\n        },\n        "counts": {\n            "oriented_signatures": len(library_records),\n            "families": {\n                f"({a},{b})": count\n                for (a, b), count in sorted(family_signature_hist.items())\n            },\n            "family_occurrences": {\n                f"({a},{b})": count\n                for (a, b), count in sorted(family_occurrence_hist.items())\n            },\n            "invariant_dimension_histogram": {\n                str(k): v for k, v in sorted(invariant_dimension_hist.items())\n            },\n            "joint_channels": sum(len(record["channels"]) for record in library_records),\n            "distinct_joint_casimir_histories": len(channel_history_hist),\n            "signatures_with_cut3_determinant_channel": determinant_cut3_signature_count,\n            "serialized_channel_kernel_nonzero_entries": total_channel_kernel_terms,\n        },\n        "joint_casimir_history_occurrence_histogram": {\n            ",".join(history): count\n            for history, count in sorted(channel_history_hist.items())\n        },\n        "signatures": library_records,\n    }\n\n    library_path = OUT / "y4_su4_exceptional_local_library.json"\n    library_path.write_text(\n        json.dumps(output_library, indent=2, sort_keys=True),\n        encoding="utf-8",\n    )\n    library_sha = sha256(library_path)\n\n    summary = {\n        "version": VERSION,\n        "status": "PASS",\n        "inputs": {\n            "catalog": str(catalog_path),\n            "catalog_sha256": sha256(catalog_path),\n            "extraction": extraction,\n        },\n        "counts": output_library["counts"],\n        "family_signature_histogram": {\n            f"({a},{b})": count\n            for (a, b), count in sorted(family_signature_hist.items())\n        },\n        "family_occurrence_histogram": {\n            f"({a},{b})": count\n            for (a, b), count in sorted(family_occurrence_hist.items())\n        },\n        "gates": {\n            "exact_gram_rank": True,\n            "prefix_casimir_closure": True,\n            "prefix_casimir_self_adjoint": True,\n            "nested_casimirs_commute": True,\n            "joint_projectors_idempotent": True,\n            "joint_projectors_orthogonal": True,\n            "joint_projectors_complete": True,\n            "charge_conjugation_consistent": True,\n            "passed": True,\n        },\n        "output": {\n            "local_library": str(library_path),\n            "local_library_sha256": library_sha,\n        },\n        "next_stage": (\n            "Use this library only on the 312 exceptional assignments. Keep every balanced "\n            "local node in the existing walled-Brauer library. Build the 76-word hybrid corpus, "\n            "sum local channel Casimirs at cuts 1,2,3, apply the unchanged fourth-order folded "\n            "coefficient, contract the exact delta-epsilon tensor networks, and add the resulting "\n            "kernel delta to the balanced N=4 evaluation."\n        ),\n        "elapsed_seconds": time.time() - started,\n    }\n\n    json_path = OUT / "SU4_LOCAL_ALGEBRA_V1.json"\n    json_path.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")\n    json_sha = sha256(json_path)\n\n    md = fr"""# SU(4) exceptional local Haar and Casimir algebra\n\n**Status:** PASS  \n**Version:** `{VERSION}`\n\n## Exact local invariant spaces\n\nThe exceptional signatures contain only:\n\n- `(4,0)` and `(0,4)`: one-dimensional epsilon invariant;\n- `(5,1)` and `(1,5)`: four-dimensional delta-epsilon invariant space.\n\nThe five natural delta-epsilon tensors have one exact linear dependency, the\nfive-index antisymmetry identity in four dimensions.\n\n## Exact channel algebra\n\nFor every oriented signature, the certificate verifies:\n\n- exact Gram rank;\n- closure of all three prefix Casimirs;\n- self-adjointness in the invariant Gram metric;\n- mutual commutativity of the nested Casimirs;\n- exact joint eigenspace decomposition;\n- projector idempotence and orthogonality;\n- completeness of the channel projectors;\n- charge-conjugation consistency.\n\nCounts:\n\n```text\noriented signatures: {len(library_records)}\njoint channels: {output_library[\'counts\'][\'joint_channels\']}\ndistinct Casimir histories: {output_library[\'counts\'][\'distinct_joint_casimir_histories\']}\nsignatures with a cut-3 determinant channel:\n    {determinant_cut3_signature_count}\n```\n\nThe local channel tensor exported for the next stage is\n\n\\[\nP_{{\\alpha}}(r,c)=\\sum_{{a,b}}K^{{(\\alpha)}}_{{ab}}T_a(r)T_b(c),\n\\]\n\nwhere the basis tensors `T_a` are explicit epsilon or delta-epsilon tensors and\neach channel carries its exact three-cut Casimir history.\n\n## Next stage\n\nKeep the verified balanced local library unchanged. Apply the exported finite\n`SU(4)` library only to the 312 exceptional assignments in the 76 mixed words,\ncontract their exact tensor networks, and add the resulting kernel correction\nto the balanced `N=4` evaluation.\n"""\n    md_path = OUT / "SU4_LOCAL_ALGEBRA_V1.md"\n    md_path.write_text(md, encoding="utf-8")\n    md_sha = sha256(md_path)\n\n    bundle = BASE / "SU4_LOCAL_ALGEBRA_V1_BUNDLE.zip"\n    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as zf:\n        for p in (json_path, md_path, library_path):\n            zf.write(p, arcname=p.name)\n        source_path = Path(globals().get("__file__", ""))\n        if source_path.is_file():\n            zf.write(source_path, arcname=source_path.name)\n\n    print("\\n" + "=" * 116)\n    print("SU(4) LOCAL ALGEBRA STATUS: PASS")\n    print("=" * 116)\n    print("oriented signatures                    :", len(library_records))\n    print("family signature histogram             :", dict(sorted(family_signature_hist.items())))\n    print("invariant dimension histogram          :", dict(sorted(invariant_dimension_hist.items())))\n    print("joint channels                         :", output_library["counts"]["joint_channels"])\n    print("distinct joint Casimir histories       :", len(channel_history_hist))\n    print("signatures with cut-3 determinant      :", determinant_cut3_signature_count)\n    print("serialized channel-kernel nonzero terms:", total_channel_kernel_terms)\n    print("JSON:", json_path, json_sha)\n    print("MD:  ", md_path, md_sha)\n    print("LIB: ", library_path, library_sha)\n    print("ZIP: ", bundle, sha256(bundle))\n    print("=" * 116)\n\n\nif __name__ == "__main__":\n    main()\n'

ENUM_SCRIPT = WORK / "y4_su4_exceptional_enumerator_v1.py"
ALG_SCRIPT = WORK / "ENGINE_Y4_su4_local_algebra_v1.py"
ENUM_SCRIPT.write_text(ENUM_SOURCE, encoding="utf-8")
ALG_SCRIPT.write_text(ALG_SOURCE, encoding="utf-8")


class Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()
        return len(data)
    def flush(self):
        for stream in self.streams:
            stream.flush()


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def upload_source_bundle() -> Path:
    existing = sorted(BASE.glob("Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE*.zip"))
    if existing:
        print("Using existing source bundle:", existing[0])
        return existing[0]

    from google.colab import files
    print("\nUPLOAD ONE FILE:")
    print("  Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE_2026-06-14_V2*.zip")
    uploaded = files.upload()
    candidates = []
    for name, data in uploaded.items():
        target = BASE / Path(name).name
        target.write_bytes(data)
        if target.name.startswith("Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE") and target.suffix == ".zip":
            candidates.append(target)
    if not candidates:
        raise FileNotFoundError("The required full symbolic source ZIP was not uploaded.")
    print("Saved source bundle:", candidates[0])
    return candidates[0]


def run_script(path: Path, label: str) -> None:
    print("\n" + "=" * 120)
    print(label)
    print("=" * 120)
    old_argv = sys.argv[:]
    try:
        sys.argv = [str(path)]
        runpy.run_path(str(path), run_name="__main__")
    finally:
        sys.argv = old_argv


def add_tree(zf: zipfile.ZipFile, root: Path, arc_root: str) -> None:
    if not root.exists():
        return
    if root.is_file():
        zf.write(root, arcname=f"{arc_root}/{root.name}")
        return
    for p in sorted(root.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=f"{arc_root}/{p.relative_to(root)}")


def main() -> None:
    started = time.time()

    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    source_zip = upload_source_bundle()

    with LOG_PATH.open("w", encoding="utf-8") as log_file:
        tee = Tee(sys.stdout, log_file)
        with contextlib.redirect_stdout(tee), contextlib.redirect_stderr(tee):
            print("SU(4) PERSISTENT PIPELINE V1")
            print("Source bundle:", source_zip)
            print("Source SHA-256:", sha256(source_zip))

            run_script(ENUM_SCRIPT, "STAGE 1 — SU(4) EXCEPTIONAL ENUMERATOR")

            enum_bundle = BASE / "SU4_EXCEPTIONAL_ENUMERATOR_V1_BUNDLE.zip"
            enum_dir = BASE / "SU4_EXCEPTIONAL_ENUMERATOR_V1"
            if not enum_bundle.exists():
                raise FileNotFoundError(f"Enumerator did not create {enum_bundle}")
            print("Enumerator bundle:", enum_bundle, sha256(enum_bundle))

            run_script(ALG_SCRIPT, "STAGE 2 — SU(4) LOCAL ALGEBRA")

            alg_bundle = BASE / "SU4_LOCAL_ALGEBRA_V1_BUNDLE.zip"
            alg_dir = BASE / "SU4_LOCAL_ALGEBRA_V1"
            if not alg_bundle.exists():
                raise FileNotFoundError(f"Local algebra did not create {alg_bundle}")
            print("Local algebra bundle:", alg_bundle, sha256(alg_bundle))

            manifest = {
                "version": "2026-06-14-su4-persistent-pipeline-v1",
                "status": "PASS",
                "source": {
                    "path": str(source_zip),
                    "sha256": sha256(source_zip),
                },
                "outputs": {
                    "SU4_EXCEPTIONAL_ENUMERATOR_V1_BUNDLE.zip": sha256(enum_bundle),
                    "SU4_LOCAL_ALGEBRA_V1_BUNDLE.zip": sha256(alg_bundle),
                },
                "elapsed_seconds": time.time() - started,
            }
            manifest_path = WORK / "SU4_PERSISTENT_PIPELINE_V1_MANIFEST.json"
            manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")

            master_zip = BASE / "SU4_PERSISTENT_RESULTS_2026-06-14.zip"
            with zipfile.ZipFile(master_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
                zf.write(enum_bundle, arcname=enum_bundle.name)
                zf.write(alg_bundle, arcname=alg_bundle.name)
                add_tree(zf, enum_dir, "SU4_EXCEPTIONAL_ENUMERATOR_V1")
                add_tree(zf, alg_dir, "SU4_LOCAL_ALGEBRA_V1")
                zf.write(ENUM_SCRIPT, arcname=f"source/{ENUM_SCRIPT.name}")
                zf.write(ALG_SCRIPT, arcname=f"source/{ALG_SCRIPT.name}")
                zf.write(manifest_path, arcname=manifest_path.name)
                zf.write(LOG_PATH, arcname=LOG_PATH.name)

            drive_dir = Path("/content/drive/MyDrive/SU4_PERSISTENT_RESULTS_2026-06-14")
            drive_dir.mkdir(parents=True, exist_ok=True)

            copies = [enum_bundle, alg_bundle, master_zip, manifest_path, LOG_PATH]
            for src in copies:
                dst = drive_dir / src.name
                shutil.copy2(src, dst)
                print("Persisted to Drive:", dst)

            print("\n" + "=" * 120)
            print("SU(4) PERSISTENT PIPELINE STATUS: PASS")
            print("=" * 120)
            print("MASTER ZIP:", master_zip)
            print("MASTER SHA-256:", sha256(master_zip))
            print("DRIVE COPY:", drive_dir / master_zip.name)
            print("The Drive copy survives Colab runtime resets.")
            print("=" * 120)

    from google.colab import files
    files.download(str(BASE / "SU4_PERSISTENT_RESULTS_2026-06-14.zip"))


if __name__ == "__main__":
    main()
